# Week 3: Small Worlds — Assignment

**Learning objectives** — In this assignment you will:

- Implement sampled average path length using BFS primitives
- Generate Erdos-Renyi random graphs as a null model
- Compute the small-world sigma coefficient by reusing your own functions
- Build the Watts-Strogatz ring-lattice-and-rewire algorithm from scratch
- Sweep over rewiring probabilities and collect C/L statistics using your own implementations
- Implement greedy routing on Kleinberg grids from scratch

## Grading

| Section | Function | Points |
|---------|----------|--------|
| 1 | `average_path_length(G, sample_size)` | 10 |
| 2 | `er_graph(n, avg_degree)` | 10 |
| 3 | `small_world_sigma(G, n_random)` | 15 |
| 4 | `generate_ws(n, k, p)` | 20 |
| 5 | `ws_sweep(n, k, p_values)` | 15 |
| 6 | `greedy_route(G, source, target, pos)` | 15 |
| — | Written questions | 15 |
| | **Total** | **100** |

## Before You Start

This assignment builds on the Week 3 lab. Make sure you are comfortable with:

- **Average shortest path length (APL)** — computed via BFS from every node; measures typical separation (Lab Section 2)
- **Small-world property** — high clustering AND short paths relative to a random baseline (Lab Section 4)
- **Watts-Strogatz model** — ring lattice + rewiring; the C(p)/L(p) vs p curve and its asymmetry (Lab Sections 5-6)
- **ER model** — random graph with fixed edge probability; serves as a null model for comparison (Lab Section 3)
- **Kleinberg model** — grid with distance-dependent long-range links; enables efficient decentralized search (Lab Section 7)

### Implementation constraints

The following functions are **banned** from your solutions:

| Banned function | Sections |
|----------------|----------|
| `nx.average_shortest_path_length` | 1, 3, 5 |
| `nx.sigma`, `nx.omega` | 3 |
| `nx.connected_watts_strogatz_graph`, `nx.watts_strogatz_graph`, `models.watts_strogatz` | 4, 5 |
| `models.greedy_route`, `netsci.models.greedy_route` | 6 |

You **may** use basic graph primitives: `G.neighbors()`, `G.nodes()`, `G.edges()`, `G.degree()`, `G.add_node()`, `G.add_edge()`, `nx.Graph()`, `nx.average_clustering()`, `nx.single_source_shortest_path_length()` (Section 1 only), and `collections.deque`.

**Important**: Later sections build on earlier ones. You must **reuse your own implementations**:
- Section 3 must use your `average_path_length` from Section 1 and your `er_graph` from Section 2
- Section 5 must use your `generate_ws` from Section 4 and your `average_path_length` from Section 1

In [1]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from netsci.loaders import load_graph
from netsci.utils import SEED
from netsci import models

In [2]:
G_karate = load_graph("karate")
G_air = load_graph("airports")

karate: 34 nodes, 78 edges (undirected)
airports: 500 nodes, 2980 edges (undirected)


---
## Section 1: Average Path Length (10 pts)

Computing the exact average path length requires BFS from every node — O(n(n+m)) which can be slow on large graphs.

Implement a **sampled** version: pick `sample_size` random source nodes, run BFS from each,
and average all the distances.

**Implementation details:**
- Use `nx.single_source_shortest_path_length(G, node)` to get distances from a source node
- If `sample_size` is None, use all nodes (exact APL)
- If `sample_size` is given, sample that many source nodes using `np.random.default_rng(SEED).choice()`
- The graph must be connected
- **Do NOT** call `nx.average_shortest_path_length`

In [3]:
def average_path_length(G, sample_size=None):
    """Compute the (sampled) average shortest path length.

    Do NOT call nx.average_shortest_path_length.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    sample_size : int or None
        If None, use all nodes (exact). Otherwise sample this many
        source nodes using np.random.default_rng(SEED).choice().

    Returns
    -------
    float
    """
    if not nx.is_connected(G):
        raise ValueError("Graph must be connected")

    nodes = list(G.nodes())
    n = len(nodes)
    if n <= 1:
        return 0.0

    if sample_size is None:
        sources = nodes
    else:
        rng = np.random.default_rng(SEED)
        sample_size = min(sample_size, n)
        sources = list(rng.choice(nodes, size=sample_size, replace=False))

    total_dist = 0.0
    for s in sources:
        lengths = nx.single_source_shortest_path_length(G, s)
        total_dist += sum(lengths.values())

    # Each source contributes distances to (n-1) other nodes
    return total_dist / (len(sources) * (n - 1))

In [4]:
# --- Validation ---
# Exact computation on karate should match NetworkX
_apl = average_path_length(G_karate, sample_size=None)
_expected = nx.average_shortest_path_length(G_karate)
assert abs(_apl - _expected) < 1e-6, f"Got {_apl}, expected {_expected}"
print(f"Karate APL (exact): {_apl:.4f} (expected: {_expected:.4f})")

# Sampled should be close
_apl_sampled = average_path_length(G_air, sample_size=50)
_apl_exact = nx.average_shortest_path_length(G_air)
assert abs(_apl_sampled - _apl_exact) < 0.3, (
    f"Sampled {_apl_sampled} too far from exact {_apl_exact}"
)
print(f"Airports APL (sampled 50): {_apl_sampled:.2f} (exact: {_apl_exact:.2f})")

# Exact on airports should match precisely
_apl_air_exact = average_path_length(G_air, sample_size=None)
assert abs(_apl_air_exact - _apl_exact) < 1e-6, (
    f"Exact airports APL: {_apl_air_exact:.4f}, expected {_apl_exact:.4f}"
)
print(f"Airports APL (exact): {_apl_air_exact:.4f} (expected: {_apl_exact:.4f})")
print("Section 1 passed!")

Karate APL (exact): 2.4082 (expected: 2.4082)


Airports APL (sampled 50): 3.01 (exact: 2.99)
Airports APL (exact): 2.9910 (expected: 2.9910)
Section 1 passed!


---
## Section 2: ER Graph (10 pts)

Generate an Erdos-Renyi G(n, p) model. Compute the edge probability as `p = avg_degree / (n - 1)`.

You may use `nx.erdos_renyi_graph` for this section. Use `seed=seed` for reproducibility.

In [15]:
def er_graph(n, avg_degree, seed=SEED):
    """Generate an ER random graph.

    Parameters
    ----------
    n : int
    avg_degree : float
    seed : int — random seed

    Returns
    -------
    nx.Graph
    """
    if n <= 0:
        return nx.Graph()

    p = avg_degree / (n - 1)
    return nx.erdos_renyi_graph(n, p, seed=seed)

In [16]:
# --- Validation ---
_g = er_graph(500, 6)
assert isinstance(_g, nx.Graph)
assert _g.number_of_nodes() == 500
_avg_deg = 2 * _g.number_of_edges() / _g.number_of_nodes()
assert abs(_avg_deg - 6) < 1.5, f"Avg degree {_avg_deg} too far from 6"

# Reproducibility: same seed should give same graph
_g2 = er_graph(500, 6)
assert _g.number_of_edges() == _g2.number_of_edges(), (
    "Same seed should produce same graph"
)

# Low clustering (ER property)
_C_er = nx.average_clustering(_g)
assert _C_er < 0.05, f"ER clustering should be low, got {_C_er:.3f}"

print(f"ER(500, 6): avg degree = {_avg_deg:.2f}, clustering = {_C_er:.4f}")
print("Section 2 passed!")

ER(500, 6): avg degree = 5.96, clustering = 0.0111
Section 2 passed!


---
## Section 3: Small-World Sigma (15 pts)

Compute the small-world coefficient:

$$\sigma = \frac{C_{real} / C_{random}}{L_{real} / L_{random}}$$

Generate `n_random` Erdos-Renyi graphs with the same n and edge probability,
compute their average C and L, and use those as the baseline.

**Implementation requirements:**
- **Reuse your own** `average_path_length` from Section 1 (do NOT call `nx.average_shortest_path_length`)
- **Reuse your own** `er_graph` from Section 2 (do NOT call `nx.erdos_renyi_graph` directly)
- **Do NOT** call `nx.sigma` or `nx.omega`
- Use `nx.average_clustering(G)` for clustering coefficients
- Use `SEED + i` as the seed for the i-th random graph (i = 0, 1, ..., n_random-1)
- Skip disconnected random graphs (only average over connected ones)

In [7]:
def small_world_sigma(G, n_random=5):
    """Compute the small-world sigma coefficient.

    Reuse your average_path_length() from Section 1 and your er_graph()
    from Section 2.  Do NOT call nx.average_shortest_path_length,
    nx.sigma, nx.omega, or nx.erdos_renyi_graph.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    n_random : int
        Number of random graphs to average over.

    Returns
    -------
    float
    """
    if not nx.is_connected(G):
        raise ValueError("Graph must be connected")

    n = G.number_of_nodes()
    if n <= 1:
        return 0.0

    C_real = nx.average_clustering(G)
    L_real = average_path_length(G, sample_size=None)

    avg_C_rand = 0.0
    avg_L_rand = 0.0
    count = 0

    avg_deg = 2 * G.number_of_edges() / n
    for i in range(n_random):
        G_rand = er_graph(n, avg_deg, seed=SEED + i)
        if not nx.is_connected(G_rand):
            continue
        avg_C_rand += nx.average_clustering(G_rand)
        avg_L_rand += average_path_length(G_rand, sample_size=None)
        count += 1

    if count == 0:
        raise ValueError("No connected random graphs generated")

    avg_C_rand /= count
    avg_L_rand /= count

    return (C_real / avg_C_rand) / (L_real / avg_L_rand)

In [8]:
# --- Validation ---
_sigma = small_world_sigma(G_air, n_random=5)
assert isinstance(_sigma, float)
# Airports should be a small world
assert _sigma > 5.0, f"Expected sigma >> 1 for airports, got {_sigma}"
assert _sigma < 100.0, f"Sigma unreasonably high: {_sigma}"

# Verify it uses a reasonable baseline: sigma should be in the right ballpark
# (known value is ~20-25 for airports with 5 random baselines)
assert _sigma > 10.0, (
    f"Sigma = {_sigma:.1f} seems too low — check that you're computing "
    f"C_real/C_rand and L_real/L_rand correctly"
)
print(f"Airports sigma = {_sigma:.2f} (> 1 = small world)")
print("Section 3 passed!")

Airports sigma = 24.06 (> 1 = small world)
Section 3 passed!


## Section 4: Watts-Strogatz Generation from Scratch (15 pts)

Implement the Watts-Strogatz model **from scratch**. **Do NOT** call `nx.watts_strogatz_graph`, `nx.connected_watts_strogatz_graph`, or `models.watts_strogatz`.

The algorithm:

1. **Build a ring lattice**: Create a graph with `n` nodes arranged in a ring. Connect each node to its `k//2` nearest neighbors on each side (total degree = k per node). Nodes should be labeled `0, 1, ..., n-1`.
2. **Rewire edges**: For each node `u` (in order `0, 1, ..., n-1`), consider each edge `(u, v)` where `v` is one of the `k//2` clockwise neighbors (i.e., `v = (u+j) % n` for `j = 1, ..., k//2`). With probability `p`, rewire it: remove `(u, v)` and add `(u, w)` where `w` is chosen uniformly at random from all nodes that are not `u` and not already a neighbor of `u`. If no valid `w` exists, skip.
3. Use `np.random.default_rng(seed)` for all randomness (both the rewire coin flip and the choice of `w`).

The resulting graph should be connected for the parameters we test.

In [9]:
def generate_ws(n, k, p, seed=SEED):
    """Generate a Watts-Strogatz graph from scratch.

    Do NOT call nx.watts_strogatz_graph, nx.connected_watts_strogatz_graph,
    or models.watts_strogatz.

    Parameters
    ----------
    n : int  — number of nodes
    k : int  — each node connected to k nearest neighbors (must be even)
    p : float — rewiring probability
    seed : int — random seed

    Returns
    -------
    nx.Graph
    """
    if n <= 0:
        return nx.Graph()
    if k % 2 != 0:
        raise ValueError("k must be even")

    G = nx.Graph()
    G.add_nodes_from(range(n))

    # Build ring lattice
    half_k = k // 2
    for u in range(n):
        for j in range(1, half_k + 1):
            v = (u + j) % n
            G.add_edge(u, v)

    rng = np.random.default_rng(seed)

    # Rewire edges
    for u in range(n):
        for j in range(1, half_k + 1):
            v = (u + j) % n
            # Only consider the original clockwise edge (u, v)
            if not G.has_edge(u, v):
                continue
            if rng.random() < p:
                # Remove current edge
                G.remove_edge(u, v)

                # Find a new node w to connect to
                forbidden = set(G.neighbors(u)) | {u, v}
                candidates = [w for w in range(n) if w not in forbidden]
                if not candidates:
                    # Restore original edge if no candidate found
                    G.add_edge(u, v)
                    continue
                w = rng.choice(candidates)
                G.add_edge(u, w)

    return G

In [10]:
# --- Validation ---
_ws = generate_ws(100, 6, 0.1)
assert isinstance(_ws, nx.Graph)
assert _ws.number_of_nodes() == 100
assert nx.is_connected(_ws), "Graph should be connected for these parameters"

# p=0 should give a regular ring lattice where every node has degree k
_ws0 = generate_ws(50, 4, 0.0)
_degrees_0 = [d for _, d in _ws0.degree()]
assert all(d == 4 for d in _degrees_0), "p=0 should give uniform degree k"

# p=0 ring lattice: check that edges are only to nearest neighbors
for u in _ws0.nodes():
    for v in _ws0.neighbors(u):
        dist = min(abs(u - v), 50 - abs(u - v))  # ring distance
        assert dist <= 2, (
            f"p=0: edge ({u},{v}) has ring distance {dist}, expected <= k//2=2"
        )

# p=0 should have high clustering (ring lattice property)
_C0 = nx.average_clustering(_ws0)
assert _C0 > 0.4, f"p=0 clustering should be high, got {_C0:.3f}"

# p=1 should have low clustering (near-random)
_ws1 = generate_ws(100, 6, 1.0)
_C1 = nx.average_clustering(_ws1)
assert _C1 < 0.2, f"p=1 clustering should be low, got {_C1:.3f}"

# Number of edges should be preserved (rewiring doesn't add/remove edges)
assert _ws0.number_of_edges() == 50 * 4 // 2, (
    f"Expected {50 * 4 // 2} edges, got {_ws0.number_of_edges()}"
)
print(f"p=0.0: C={_C0:.3f}, edges={_ws0.number_of_edges()}")
print(f"p=1.0: C={_C1:.3f}, edges={_ws1.number_of_edges()}")
print("Section 4 passed!")

p=0.0: C=0.500, edges=100
p=1.0: C=0.043, edges=300
Section 4 passed!


---
## Section 5: WS Sweep (15 pts)

Sweep over a list of rewiring probabilities and collect the average clustering coefficient
and average shortest path length at each p.

**Implementation requirements:**
- **Reuse your own** `generate_ws` from Section 4 (do NOT call `nx.watts_strogatz_graph` or `models.watts_strogatz`)
- Use `nx.average_clustering(G)` for clustering
- **Do NOT** call `nx.average_shortest_path_length` — use your own `average_path_length` from Section 1

Return a dictionary with keys:
- `"p"` — list of p values
- `"clustering"` — list of average clustering coefficients
- `"path_length"` — list of average shortest path lengths

In [11]:
def ws_sweep(n, k, p_values):
    """Sweep over p values and collect WS graph statistics.

    Reuse your generate_ws() from Section 4 and your average_path_length()
    from Section 1.  Do NOT call nx.watts_strogatz_graph,
    models.watts_strogatz, or nx.average_shortest_path_length.

    Parameters
    ----------
    n : int
    k : int
    p_values : list of float

    Returns
    -------
    dict with keys 'p', 'clustering', 'path_length'
    """
    result = {
        "p": [],
        "clustering": [],
        "path_length": [],
    }

    for p in p_values:
        G = generate_ws(n, k, p)
        result["p"].append(p)
        result["clustering"].append(nx.average_clustering(G))
        result["path_length"].append(average_path_length(G, sample_size=None))

    return result

In [18]:
# --- Validation ---
_ps = [0.0, 0.01, 0.1, 1.0]
_result = ws_sweep(100, 6, _ps)

assert set(_result.keys()) == {"p", "clustering", "path_length"}
assert len(_result["p"]) == 4
assert len(_result["clustering"]) == 4
assert len(_result["path_length"]) == 4

# Clustering should decrease with p
assert _result["clustering"][0] > _result["clustering"][-1], (
    "Clustering should be higher at p=0 than p=1"
)

# Path length should decrease with p
assert _result["path_length"][0] > _result["path_length"][-1], (
    "Path length should be higher at p=0 than p=1"
)

# p=0 clustering for k=6 ring lattice should be exactly 0.6
assert abs(_result["clustering"][0] - 0.6) < 0.05, (
    f"p=0 clustering should be ~0.6 for k=6, got {_result['clustering'][0]:.3f}"
)

# p=0 path length for n=100, k=6 should be ~8.5 (ring lattice)
assert _result["path_length"][0] > 5.0, (
    f"p=0 path length seems too short: {_result['path_length'][0]:.2f}"
)

# The small-world regime: at p=0.01 path length should drop significantly
# while clustering stays high
_L_ratio = _result["path_length"][1] / _result["path_length"][0]
_C_ratio = _result["clustering"][1] / _result["clustering"][0]
assert _L_ratio < _C_ratio, (
    "Small-world asymmetry: L should drop faster than C at p=0.01"
)

print(f"p=0.0:  C={_result['clustering'][0]:.3f}  L={_result['path_length'][0]:.2f}")
print(f"p=0.01: C={_result['clustering'][1]:.3f}  L={_result['path_length'][1]:.2f}")
print(f"p=0.33: C={_result['clustering'][2]:.3f}  L={_result['path_length'][2]:.2f}")
print(f"p=1.0:  C={_result['clustering'][-1]:.3f}  L={_result['path_length'][-1]:.2f}")
print("Section 5 passed!")

p=0.0:  C=0.600  L=8.76
p=0.01: C=0.592  L=7.27
p=0.33: C=0.462  L=3.81
p=1.0:  C=0.043  L=2.72
Section 5 passed!


## Section 6: Greedy Routing on Kleinberg Grid (15 pts)

Implement greedy routing on a Kleinberg grid **from scratch**. **Do NOT** call `models.greedy_route` or `netsci.models.greedy_route`.

At each step, forward the message to the neighbor that is **closest to the target** by Manhattan distance:

$$d((r_1, c_1), (r_2, c_2)) = |r_1 - r_2| + |c_1 - c_2|$$

Keep track of **visited nodes** to avoid cycles — do not revisit a node already on the path.

The function receives a graph `G` whose nodes are `(row, col)` tuples, a `source` node,
a `target` node, and a `pos` dict mapping nodes to coordinates.

Return the path as a list of nodes from source to target (inclusive), or `None` if the
greedy algorithm gets stuck (no unvisited neighbors).

In [13]:
def greedy_route(G, source, target, pos):
    """Greedy routing on a grid graph.

    Do NOT call models.greedy_route or netsci.models.greedy_route.

    Parameters
    ----------
    G : nx.Graph — nodes are (row, col) tuples
    source : tuple — start node
    target : tuple — destination node
    pos : dict — {node: (row, col)} coordinates

    Returns
    -------
    list of nodes (path from source to target) or None if stuck
    """
    if source == target:
        return [source]

    def manhattan(u, v):
        return abs(u[0] - v[0]) + abs(u[1] - v[1])

    visited = {source}
    path = [source]
    current = source

    while current != target:
        neighbors = [n for n in G.neighbors(current) if n not in visited]
        if not neighbors:
            return None
        # choose neighbor closest to target by Manhattan distance
        best = min(neighbors, key=lambda n: manhattan(pos[n], pos[target]))
        visited.add(best)
        path.append(best)
        current = best

    return path

In [14]:
# --- Validation ---
_G_k, _pos_k = models.kleinberg_grid(n=10, r=2, q=1, seed=SEED)
_nodes_k = list(_G_k.nodes())
_path = greedy_route(_G_k, _nodes_k[0], _nodes_k[-1], _pos_k)
assert _path is not None, "Greedy route should find a path on r=2 grid"
assert isinstance(_path, list)
assert _path[0] == _nodes_k[0], "Path should start at source"
assert _path[-1] == _nodes_k[-1], "Path should end at target"
# Path should be shorter than grid diameter
assert len(_path) < _G_k.number_of_nodes(), "Path too long"
# Each step should be along an edge
for i in range(len(_path) - 1):
    assert _G_k.has_edge(_path[i], _path[i + 1]), (
        f"No edge between {_path[i]} and {_path[i + 1]}"
    )
# No repeated nodes (greedy should track visited)
assert len(_path) == len(set(_path)), "Path contains repeated nodes — track visited!"

# Verify greedy property: each step should move closer to or maintain
# distance to target (by Manhattan distance)
for i in range(len(_path) - 1):
    _d_curr = abs(_path[i][0] - _nodes_k[-1][0]) + abs(_path[i][1] - _nodes_k[-1][1])
    _d_next = abs(_path[i+1][0] - _nodes_k[-1][0]) + abs(_path[i+1][1] - _nodes_k[-1][1])
    # The chosen neighbor should be the closest unvisited neighbor to target
    _visited_so_far = set(_path[:i+1])
    _unvisited_neighbors = [n for n in _G_k.neighbors(_path[i]) if n not in _visited_so_far]
    if _unvisited_neighbors:
        _best_dist = min(
            abs(n[0] - _nodes_k[-1][0]) + abs(n[1] - _nodes_k[-1][1])
            for n in _unvisited_neighbors
        )
        assert _d_next == _best_dist, (
            f"Step {i}: chose neighbor at distance {_d_next}, "
            f"but closest unvisited was at {_best_dist}"
        )

# Test source == target edge case
_path_self = greedy_route(_G_k, _nodes_k[0], _nodes_k[0], _pos_k)
assert _path_self == [_nodes_k[0]], "source==target should return [source]"

print(f"Greedy path length: {len(_path) - 1} hops")
print("Section 6 passed!")

Greedy path length: 11 hops
Section 6 passed!


---
## Written Questions (15 pts)

### Question 1 (5 pts)

In the Watts-Strogatz model, why does a **tiny amount of rewiring** (e.g., p=0.01) dramatically
reduce the average path length, but barely affect the clustering coefficient?

**Use specific numerical evidence from your `ws_sweep` results** (Section 5) to support your answer. Quote the C and L values at p=0 and p=0.01, and explain the asymmetry in terms of the network structure.

*Hint: Think about the difference between a global property (path length) and a local property (clustering) when a few edges are rewired.*

**Your Answer:**

La p=0 (ring lattice) obtinem aproximativ **C ≈ 0.600** si **L ≈ 8.76**. La p=0.01, clusteringul ramane tot mare (cam **C ≈ 0.592**), dar lungimea medie scade destul de mult la **L ≈ 7.27**.

Explicatia e ca clusteringul e o proprietate locala: daca schimbi doar cateva muchii, majoritatea nodurilor pastreaza aceiasi vecini si aceleasi triunghiuri, deci C ramane aproape la fel. In schimb distanta medie e o proprietate globala: chiar si putine scurtaturi (muchiile rewire-uite) reduc foarte mult numarul de salturi dintre noduri indepartate. Asa se explica de ce L cade rapid, iar C se schimba putin.


### Question 2 (4 pts)

The airports network has sigma > 1, confirming it is a small world.
Does this mean "six degrees of separation" literally holds for airports?

**Reference the specific APL and sigma values** you computed in Section 3 and explain what sigma captures versus what "six degrees" claims. What does "a degree of separation" mean in the airport context?

*Hint: Consider the difference between the structural property (sigma) and the absolute path length.*

**Your Answer:**

Nu, sigma > 1 nu inseamna ca in mod literal exista "6 grade" de separare la aeroporturi. Sigma arata ca reteaua e small-world (clustering mult mai mare decat un graf random cu aceeasi densitate si drumuri relativ scurte), dar nu spune un numar fix de hopuri.

In Sectiunea 3 am gasit **APL ≈ 2.99** si **σ ≈ 24.06**. APL inseamna cate zboruri (hopuri) e, in medie, intre doua aeroporturi alese la intamplare. Un "grad de separare" aici e un hop (un segment de zbor).

Asadar, sigma e despre comparatia structurala cu un graf random, iar "6 grade" e despre numarul absolut de hopuri. La aeroporturi se ajunge, in medie, in cam 3 hopuri, nu 6.


### Question 3 (6 pts)

Kleinberg proved that greedy routing is efficient only when the clustering exponent r equals the grid dimension d. 

(a) Explain why r=0 fails for greedy routing even though short paths exist in the graph.
(b) Explain what property of r=2 (on a 2D grid) makes greedy search efficient.
(c) If you placed a Kleinberg model on a 3D grid instead, what value of r would be optimal? Justify your answer.

*Hint: Think about the distribution of link lengths at each r value and what a greedy algorithm needs at different stages of its journey.*

**Your Answer:**

(a) La r=0, legaturile lungi sunt alese complet la intamplare. Asta inseamna ca poti sa sari oricand intr-o zona care nu te apropie de tinta, iar algoritmul greedy poate sa ramana in bucla sau sa se blocheze, chiar daca exista un drum scurt. 

(b) La r=2 pe o grila 2D, probabilitatea unei legaturi lungi scade cu distanta^2 — adica scade ca aria cercului. Astfel apar legaturi de lungimi potrivite pentru fiecare etapa: sunt si cele scurte, si cateva medii, si cateva lungi. Greedy poate atunci sa gaseasca mereu un pas care sa reduca distanta catre tinta.

(c) In 3D, optimal e r = d = 3. Motivul e acelasi: volumul unei „benzi” la distanta ℓ creste ca ℓ^{d-1}, iar daca p ~ 1/ℓ^d atunci fiecare scara de distanta e reprezentata uniform si greedy poate avansa eficient.
